In [ ]:
!pip -q install transformers accelerate sentencepiece spacy nltk
!python -m spacy download en_core_web_sm -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 141.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import json


class SimpleMediverseFormatter:
    """
    Simple, clean Mediverse-to-BioMistral formatter
    Task: Implement anatomy + timestamp encoder
    """


    def __init__(self):
        # ANATOMY ENCODER: Simple mapping dictionary
        self.anatomy_map = {
            # Foot parts
            "footskin": "foot skin",
            "metararsal1": "first metatarsal",
            "metatarsal1": "first metatarsal",
            "proximalphalanx1": "proximal phalanx of first toe",


            # General foot areas
            "footr": "right foot",
            "footl": "left foot",
            "foot": "foot",


            # Add more as needed
            "unknown": "anatomical region"
        }


        # Tool name cleanup
        self.tool_map = {
            "Scalpel": "scalpel",
            "BoneSaw": "bone saw",
            "Drill": "drill",
            "marker": "marking pen"
        }


    def encode_anatomy(self, target):
        """
        ANATOMY ENCODER: Convert technical names to medical terms
        """
        target_lower = target.lower()


        # Direct mapping
        if target_lower in self.anatomy_map:
            return self.anatomy_map[target_lower]


        # Partial matching for complex names
        for tech_name, medical_name in self.anatomy_map.items():
            if tech_name in target_lower:
                return medical_name


        # Fallback
        return f"anatomical structure"


    def encode_timestamps(self, json_data):
        """
        TIMESTAMP ENCODER: Sort by time and group similar actions
        """
        # Get data from JSON
        if 'data' in json_data:
            entries = json_data['data']
        elif 'Items' in json_data:
            entries = json_data['Items']
        else:
            entries = json_data


        # Sort by timestamp
        sorted_entries = sorted(entries, key=lambda x: float(x['timestamp']))


        # Simple grouping: combine repeated actions on same target within 5 seconds
        grouped = []
        i = 0
        while i < len(sorted_entries):
            current = sorted_entries[i]
            similar_actions = [current]


            # Look for similar actions in next few entries
            j = i + 1
            while j < len(sorted_entries):
                next_entry = sorted_entries[j]
                time_diff = float(next_entry['timestamp']) - float(current['timestamp'])


                # Group if within 5 seconds and same action/target
                if (time_diff <= 5 and
                    next_entry['action'] == current['action'] and
                    next_entry.get('target', next_entry.get('bodypart')) == current.get('target', current.get('bodypart'))):
                    similar_actions.append(next_entry)
                    j += 1
                else:
                    break


            # Create grouped entry
            if len(similar_actions) > 1:
                grouped.append({
                    'timestamp': current['timestamp'],
                    'tool': current['tool'],
                    'action': current['action'],
                    'target': current.get('target', current.get('bodypart')),
                    'count': len(similar_actions),
                    'grouped': True
                })
            else:
                grouped.append(current)


            i = j if j > i + 1 else i + 1


        return grouped


    def convert_to_medical_step(self, entry):
        """
        Convert single entry to medical step description
        """
        tool = entry['tool']
        action = entry['action']
        target = entry.get('target', entry.get('bodypart', 'unknown'))


        # Apply encoders
        medical_target = self.encode_anatomy(target)
        clean_tool = self.tool_map.get(tool, tool.lower())


        # Handle grouped actions
        if entry.get('grouped', False):
            count = entry.get('count', 1)
            if count > 1:
                if 'cutting' in action:
                    return f"Multiple incisions made on {medical_target} using {clean_tool} ({count} cuts)"
                elif 'drilling' in action:
                    return f"Multiple drilling procedures on {medical_target} using {clean_tool} ({count} holes)"
                else:
                    return f"Multiple {action} actions on {medical_target} using {clean_tool} ({count}x)"


        # Single actions
        if 'cutting_start' in action:
            return f"Incision made on {medical_target} using {clean_tool}"
        elif 'drilling_start' in action:
            return f"Drilling procedure performed on {medical_target}"
        elif 'drawing_start' in action:
            return f"Surgical site marked on {medical_target}"
        else:
            return f"Surgical procedure on {medical_target} using {clean_tool}"


    def format_for_biomistral(self, json_data):
        """
        Main function: Convert JSON to BioMistral-ready prompt
        """
        # STEP 1: Timestamp encoding
        encoded_entries = self.encode_timestamps(json_data)


        # STEP 2: Convert to medical steps
        medical_steps = []
        for entry in encoded_entries:
            step = self.convert_to_medical_step(entry)
            medical_steps.append(step)


        # STEP 3: Create BioMistral prompt with numbered bullets
        steps_text = "\n".join([f"{i + 1}. {step}" for i, step in enumerate(medical_steps)])


        prompt = f"""You are a clinical documentation AI trained on surgical protocols and biomedical literature.


Task: Convert the following surgical steps into a brief, clinical narration suitable for a surgical operative note.


Instructions:
- Do NOT invent or infer any patient details, anatomical locations, or conditions.
- Do NOT describe any complications, findings, or outcomes unless specified.
- Use only the surgical actions provided below.
- Use precise medical terminology in a concise, professional tone.


Steps:
{steps_text}


Clinical Narration:"""


        return medical_steps, prompt

In [ ]:
# import json

# class SimpleMediverseFormatter:
def test_formatter_from_file(json_filepath):
    """
    Test the formatter using data from a user-uploaded JSON file
    """
    formatter = SimpleMediverseFormatter()

    # Read JSON data from file
    with open(json_filepath, 'r') as infile:
        sample_data = json.load(infile)

    # Process with formatter
    steps, prompt = formatter.format_for_biomistral(sample_data)

    print("SIMPLE MEDIVERSE-TO-BIOMISTRAL FORMATTER TEST")

    print("\nFORMATTED MEDICAL STEPS:")
    for i, step in enumerate(steps, 1):
        print(f"{i}. {step}")

    print(f"\nBIOMISTRAL-READY PROMPT:")
    print(prompt)

    return steps, prompt

# Example usage:
if __name__ == "__main__":
    # filename
    test_formatter_from_file('CombinedToolLog.json')

SIMPLE MEDIVERSE-TO-BIOMISTRAL FORMATTER TEST

FORMATTED MEDICAL STEPS:
1. Incision made on foot skin using scalpel
2. Incision made on first metatarsal using scalpel
3. Multiple incisions made on foot skin using scalpel (4 cuts)
4. Incision made on proximal phalanx of first toe using scalpel
5. Multiple incisions made on foot skin using scalpel (2 cuts)
6. Multiple incisions made on first metatarsal using bone saw (2 cuts)
7. Drilling procedure performed on first metatarsal

BIOMISTRAL-READY PROMPT:
You are a clinical documentation AI trained on surgical protocols and biomedical literature.


Task: Convert the following surgical steps into a brief, clinical narration suitable for a surgical operative note.


Instructions:
- Do NOT invent or infer any patient details, anatomical locations, or conditions.
- Do NOT describe any complications, findings, or outcomes unless specified.
- Use only the surgical actions provided below.
- Use precise medical terminology in a concise, professiona

In [ ]:
stepspip, promptpip = test_formatter_from_file('CombinedToolLog.json')

SIMPLE MEDIVERSE-TO-BIOMISTRAL FORMATTER TEST

FORMATTED MEDICAL STEPS:
1. Incision made on foot skin using scalpel
2. Incision made on first metatarsal using scalpel
3. Multiple incisions made on foot skin using scalpel (4 cuts)
4. Incision made on proximal phalanx of first toe using scalpel
5. Multiple incisions made on foot skin using scalpel (2 cuts)
6. Multiple incisions made on first metatarsal using bone saw (2 cuts)
7. Drilling procedure performed on first metatarsal

BIOMISTRAL-READY PROMPT:
You are a clinical documentation AI trained on surgical protocols and biomedical literature.


Task: Convert the following surgical steps into a brief, clinical narration suitable for a surgical operative note.


Instructions:
- Do NOT invent or infer any patient details, anatomical locations, or conditions.
- Do NOT describe any complications, findings, or outcomes unless specified.
- Use only the surgical actions provided below.
- Use precise medical terminology in a concise, professiona

In [ ]:
promptpip

'You are a clinical documentation AI trained on surgical protocols and biomedical literature.\n\n\nTask: Convert the following surgical steps into a brief, clinical narration suitable for a surgical operative note.\n\n\nInstructions:\n- Do NOT invent or infer any patient details, anatomical locations, or conditions.\n- Do NOT describe any complications, findings, or outcomes unless specified.\n- Use only the surgical actions provided below.\n- Use precise medical terminology in a concise, professional tone.\n\n\nSteps:\n1. Incision made on foot skin using scalpel\n2. Incision made on first metatarsal using scalpel\n3. Multiple incisions made on foot skin using scalpel (4 cuts)\n4. Incision made on proximal phalanx of first toe using scalpel\n5. Multiple incisions made on foot skin using scalpel (2 cuts)\n6. Multiple incisions made on first metatarsal using bone saw (2 cuts)\n7. Drilling procedure performed on first metatarsal\n\n\nClinical Narration:'

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "BioMistral/BioMistral-7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/14.5G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
prompt = promptpip
# Generate
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=6000,
    do_sample=True,
    top_p=0.9,
    temperature=0.7
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are a clinical documentation AI trained on surgical protocols and biomedical literature.


Task: Convert the following surgical steps into a brief, clinical narration suitable for a surgical operative note.


Instructions:
- Do NOT invent or infer any patient details, anatomical locations, or conditions.
- Do NOT describe any complications, findings, or outcomes unless specified.
- Use only the surgical actions provided below.
- Use precise medical terminology in a concise, professional tone.


Steps:
1. Incision made on foot skin using scalpel
2. Incision made on first metatarsal using scalpel
3. Multiple incisions made on foot skin using scalpel (4 cuts)
4. Incision made on proximal phalanx of first toe using scalpel
5. Multiple incisions made on foot skin using scalpel (2 cuts)
6. Multiple incisions made on first metatarsal using bone saw (2 cuts)
7. Drilling procedure performed on first metatarsal


Clinical Narration: The foot was incised on the medial dorsal side, the first m

In [ ]:
text = tokenizer.decode(outputs[0], skip_special_tokens=True)
prompt_out = text.split("Clinical Narration: ", 1)[1].strip()

In [ ]:
prompt_out

'The foot was incised on the medial dorsal side, the first metatarsal was incised, and multiple incisions were made on the medial dorsal side of the foot. The proximal phalanx of the first toe was incised, and multiple incisions were made on the medial dorsal side of the foot. The first metatarsal was cut using a bone saw, and a drilling procedure was performed.'

In [ ]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")

NOMENCLATURE = {
    "drawing":       ("Marking",        ["Skin marking", "Demarcation"]),
    "incision":      ("Incision",       ["Cutaneous incision"]),
    "cutting":       ("Cutting",        ["Division"]),
    "sawing":        ("Osteotomy",      ["Sawing", "Bone cutting"]),
    "drilling":      ("Drilling",       ["Burring"]),
    "cauterization": ("Cauterization",  ["Coagulation", "Electrocautery"]),
    "irrigation":    ("Irrigation",     ["Lavage"]),
    "suction":       ("Suctioning",     ["Aspiration"]),
    "retraction":    ("Retraction",     ["Exposure"]),
    "closure":       ("Closure",        ["Wound closure", "Skin closure"]),
    "suturing":      ("Suturing",       ["Approximation"]),
    "dissection":    ("Dissection",     []),
    "removal":       ("Removal",        ["Extraction", "Explantation"]),
    "insertion":     ("Insertion",      ["Placement"]),
    "excision":      ("Excision",       ["Resection"]),
}

VERBS = {
    "drawing":       ("Marked",       ["Outlined", "Demarcated"]),
    "incision":      ("Incised",      ["Opened"]),
    "cutting":       ("Cut",          ["Divided"]),
    "sawing":        ("Sawed",        ["Osteotomized", "Cut through"]),
    "drilling":      ("Drilled",      ["Burred"]),
    "cauterization": ("Cauterized",   ["Coagulated", "Electrocauterized"]),
    "irrigation":    ("Irrigated",    ["Lavaged"]),
    "suction":       ("Suctioned",    ["Aspirated"]),
    "retraction":    ("Retracted",    ["Exposed"]),
    "closure":       ("Closed",       ["Wound closed", "Skin closed"]),
    "suturing":      ("Sutured",      ["Approximated"]),
    "dissection":    ("Dissected",    []),
    "removal":       ("Removed",      ["Extracted", "Explant"]),
    "insertion":     ("Inserted",     ["Placed"]),
    "excision":      ("Excised",      ["Resected"]),
}

KNOWN_TOOLS = {"scalpel", "bone saw", "drill", "forceps", "retractor", "suction", "irrigator"}

def detect_actions(prompt_out: str):
    doc = nlp(prompt_out)
    sentences = list(doc.sents)
    results = []
    issues = []

    for i, sent in enumerate(sentences, 1):
        text = sent.text.strip()
        action_detected = None
        tool_detected = None
        for act in NOMENCLATURE.keys():
            if re.search(rf"\b{act}\w*\b", text, re.IGNORECASE):
                action_detected = act
                break
        for act, (verb, aliases) in VERBS.items():
            if any(re.search(rf"\b{alias.lower()}\b", text.lower()) for alias in [verb]+aliases):
                action_detected = act
                break

        for tool in KNOWN_TOOLS:
            if tool in text.lower():
                tool_detected = tool
                break

        if not action_detected:
            issues.append(f"Sentence {i}: Action not recognized -> '{text}'")
        if not tool_detected:
            issues.append(f"Sentence {i}: Tool not recognized -> '{text}'")

        results.append((i, text, action_detected, tool_detected))

    return results, issues

prompt_out = prompt_out
results, issues = detect_actions(prompt_out)

print("Detections:")
for r in results:
    print(r)

print("\nPotential Hallucinations:")
for i in issues:
    print(" -", i)

Detections:
(1, 'The foot was incised on the medial dorsal side, the first metatarsal was incised, and multiple incisions were made on the medial dorsal side of the foot.', 'incision', None)
(2, 'The proximal phalanx of the first toe was incised, and multiple incisions were made on the medial dorsal side of the foot.', 'incision', None)
(3, 'The first metatarsal was cut using a bone saw, and a drilling procedure was performed.', 'cutting', 'drill')

Potential Hallucinations:
 - Sentence 1: Tool not recognized -> 'The foot was incised on the medial dorsal side, the first metatarsal was incised, and multiple incisions were made on the medial dorsal side of the foot.'
 - Sentence 2: Tool not recognized -> 'The proximal phalanx of the first toe was incised, and multiple incisions were made on the medial dorsal side of the foot.'


In [ ]:
import json
import os

def create_simple_narration_json(json_filename="CombinedToolLog.json"):
    """
    Reads a JSON log and produces a list with timestamp and narration.
    Narration is always: "<action> on <target> using <tool>."
    Missing fields stay empty.
    Also saves the output as enhanced_<filename>.json
    """
    try:
        with open(json_filename, 'r') as f:
            json_data = json.load(f)
    except Exception as e:
        print("Error loading JSON.", e)
        return []

    try:
        sorted_entries = sorted(json_data["data"], key=lambda x: float(x["timestamp"]))
    except Exception as e:
        print("Error sorting timestamps.", e)
        return []

    result = []

    for entry in sorted_entries:
        ts = entry.get("timestamp", "")
        action = entry.get("action", "")
        target = entry.get("target", "")
        tool = entry.get("tool", "")

        narration = f"{action} on {target} using {tool}."

        result.append({
            "timestamp": ts,
            "narration": narration
        })

    base = os.path.splitext(json_filename)[0]
    out_name = f"enhanced_{base}.json"

    with open(out_name, "w") as f:
        json.dump(result, f, indent=2)

    print(f"Saved to {out_name}")
    return result

pipeline = create_simple_narration_json()

Saved to enhanced_CombinedToolLog.json


In [ ]:
import json
import nltk
nltk.download('punkt_tab')
nltk.download("punkt")

def map_sentences_sequential(enhanced_json_path, narration_text):
    """
    Maps each sentence of the large narration paragraph to timestamps
    in enhanced JSON (sequential mapping).
    Produces final_mapped.json
    """
    with open(enhanced_json_path, "r") as f:
        log_entries = json.load(f)

    # Split narration text into sentences
    sentences = nltk.sent_tokenize(narration_text)
    sentences = [s.strip() for s in sentences if s.strip()]

    if len(sentences) != len(log_entries):
        print(f"[WARNING] {len(sentences)} sentences vs {len(log_entries)} timestamps")

    final_output = []

    for i, sentence in enumerate(sentences):
        timestamp = log_entries[i]["timestamp"] if i < len(log_entries) else None

        final_output.append({
            "timestamp": timestamp,
            "sentence": sentence
        })

    # Save final JSON
    with open("final_mapped.json", "w") as f:
        json.dump(final_output, f, indent=2)

    print("[2] Final mapped JSON saved as: final_mapped.json")
    return final_output

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
map_sentences_sequential("enhanced_CombinedToolLog.json",prompt_out)

[WARNING] 3 sentences vs 12 timestamps
[2] Final mapped JSON saved as: final_mapped.json


[{'timestamp': 20.760000228881836,
  'sentence': 'The foot was incised on the medial dorsal side, the first metatarsal was incised, and multiple incisions were made on the medial dorsal side of the foot.'},
 {'timestamp': 27.399999618530273,
  'sentence': 'The proximal phalanx of the first toe was incised, and multiple incisions were made on the medial dorsal side of the foot.'},
 {'timestamp': 27.899999618530273,
  'sentence': 'The first metatarsal was cut using a bone saw, and a drilling procedure was performed.'}]